# Summarization
### AI shortens long text into a few sentences!
---
**Model:** t5-small
**How:** Direct model loading (pipeline does not support summarization in new transformers)

> Give the AI a long paragraph.
> It returns a short summary!


---
## Step 1 — Install and load


In [ ]:
!pip install transformers gradio sentencepiece -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL_NAME = 't5-small'

print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('Model ready!')


---
## Step 2 — Create a simple summarize function

> T5 needs the text to start with `'summarize: '`
> That tells T5 what task to do.


In [ ]:
def summarize(text, max_words=60):
    # T5 needs 'summarize: ' at the start
    input_text = 'summarize: ' + text

    # Convert text to numbers
    inputs = tokenizer(
        input_text,
        return_tensors = 'pt',
        max_length     = 512,
        truncation     = True
    )

    # Generate the summary
    output = model.generate(
        **inputs,
        max_new_tokens  = max_words,
        min_length      = 10,
        length_penalty  = 2.0,
        num_beams       = 4,
        early_stopping  = True
    )

    # Convert numbers back to text
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    return result

print('summarize() function ready!')


---
## Step 3 — Summarise a paragraph


In [ ]:
long_text = '''
Artificial intelligence is rapidly transforming industries around the world.
From healthcare to transportation, AI systems are being used to automate tasks,
improve decision making, and discover patterns in large datasets.
Hospitals are using AI to detect diseases earlier.
Self-driving cars use AI to navigate roads safely.
Farmers are using AI to predict weather and optimise crop yields.
The technology is growing fast and experts believe AI will change
nearly every job and industry within the next ten years.
'''

summary = summarize(long_text)

print('Original:')
print(long_text.strip())
print()
print('Summary:')
print(summary)
print()
print(f'Original : {len(long_text.split())} words')
print(f'Summary  : {len(summary.split())} words')
print(f'Reduced  : {round((1 - len(summary.split())/len(long_text.split()))*100)}%')


---
## Step 4 — Summarise news articles


In [ ]:
articles = [
    '''Scientists have developed a new solar panel technology that can convert
    sunlight into electricity with 40 percent efficiency. This is double the
    rate of current commercial panels. The breakthrough could make solar energy
    much cheaper and more accessible for homes and businesses worldwide.
    Researchers say the panels could be available in the market within five years.''',

    '''A team of engineers has created a robot that can perform complex surgeries
    with greater precision than human doctors. The robot uses AI to analyse
    medical scans and plan the safest path for each operation. Clinical trials
    showed that patients recovered 30 percent faster compared to traditional surgery.
    The technology is being tested in hospitals across three countries.''',
]

for i, article in enumerate(articles, 1):
    summary = summarize(article)
    print(f'Article {i} Summary:')
    print(summary)
    print()


---
## Step 5 — Try your own text!

> ✏️ Paste any long paragraph — minimum 20 words!


In [ ]:
# ✏️ Paste any long text here!
my_text = '''
Electric vehicles are becoming more popular every year as battery technology
improves and charging stations expand across the country. Major car manufacturers
are investing billions of dollars to develop new electric models. Governments
are also offering tax incentives to encourage people to switch from petrol cars.
Experts predict that by 2035, more than half of all new cars sold globally
will be electric. This shift is expected to significantly reduce carbon emissions
and help slow the effects of climate change.
'''

summary = summarize(my_text)

print('Your summary:')
print(summary)
print()
print(f'Reduced from {len(my_text.split())} words to {len(summary.split())} words')


---
## Step 6 — Build a Gradio App

> Paste any long text.
> Choose how short you want the summary.
> AI summarises it instantly!


In [ ]:
import gradio as gr

def summarize_app(text, max_words):
    if not text.strip():
        return 'Please paste some text first!'
    if len(text.split()) < 20:
        return 'Please enter at least 20 words!'

    summary    = summarize(text, max_words=int(max_words))
    orig_count = len(text.split())
    summ_count = len(summary.split())
    reduction  = round((1 - summ_count / orig_count) * 100)

    return f'{summary}\n\n--- Stats ---\nOriginal : {orig_count} words\nSummary  : {summ_count} words\nReduced  : {reduction}%'


app = gr.Interface(
    fn      = summarize_app,
    inputs  = [
        gr.Textbox(
            label       = 'Paste your long text here',
            placeholder = 'Paste any paragraph or article...',
            lines       = 6
        ),
        gr.Slider(
            minimum = 20,
            maximum = 80,
            value   = 50,
            step    = 10,
            label   = 'Summary length (words)'
        )
    ],
    outputs     = gr.Textbox(label='Summary + Stats', lines=6),
    title       = 'AI Summarizer — T5',
    description = 'Paste any long text — AI shortens it for you!'
)

app.launch(share=True)


In [ ]:
app.close()
print('App closed.')


---
## What we learned
- T5 model is loaded directly using `AutoTokenizer` and `AutoModelForSeq2SeqLM`
- T5 needs `'summarize: '` prefix — that tells it what task to perform
- `max_words` slider controls how short the summary is
- Gradio wraps it into a shareable web app

**Tomorrow — Day 8:** We build a chatbot that remembers your conversation!
